<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c04-header.png" alt="Nextia Learning · SQL and Data Preparation for AI" width="100%">

# Solution: Fit transformations correctly

**[C04-M05-L03 · Fit transformations correctly](https://learning.nextia-ai.com/courses/sql/m05/fit-transformations-correctly/)** · C04, SQL and Data Preparation for AI · Module 5, lesson 3 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** fit imputation, scaling and one-hot encoding on the train set only, keep it that way with a scikit-learn pipeline, handle categories that train never saw, and measure how much leakage inflates a score.

| | |
|---|---|
| **Time** | About 55 minutes |
| **Needs** | An internet connection for the setup cells. No account. The first cell installs pandas 3.0.6 and scikit-learn 1.9.1. |
| **You should know** | The time split and its three files, from [Choose a split](https://learning.nextia-ai.com/courses/sql/m05/choose-a-split/). The meaning of the missing values, from [Handle messy values](https://learning.nextia-ai.com/courses/sql/m04/handle-messy-values/). |
| **You do not need** | The local project from Modules 1 to 5. The setup below builds the database, the modelling table and the split here. |
| **Tested** | 2026-10-07, Python 3.14.6 with pandas 3.0.6 and scikit-learn 1.9.1, in Jupyter on macOS, from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every script, task and check of the lesson, and the experiments that measure each leak. The [lesson page](https://learning.nextia-ai.com/courses/sql/m05/fit-transformations-correctly/) has the full explanations, the diagram and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C04/M05-L03-fit-transformations/fit-transformations-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs `pandas` 3.0.6 and `scikit-learn` 1.9.1 in the versions that the course uses. It needs an internet connection and takes up to a minute the first time.

You should see `pandas 3.0.6`, `scikit-learn 1.9.1`. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. Restarting is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==3.0.6 scikit-learn==1.9.1
import pandas; print("pandas", pandas.__version__)
import sklearn; print("scikit-learn", sklearn.__version__)

## Setup: get the practice data

Run the next cell. It downloads the three files of the Larkfield help-desk export (about 200 KB), checks each file's checksum, and builds the practice database, `larkfield.db`. It works in a folder `ticket-data`, laid out like your project from Module 1, so the code of the lesson runs here unchanged. It needs an internet connection and no account, and takes about 10 seconds.

You should see `Ready: customers 240, tickets 1417, outcomes 1874`. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings (it needs a verified phone number).
- `wrong checksum`: delete the folder `ticket-data` (in Colab: the **Files** panel on the left), then run the cell again.

This is the same database that `get_data.py` builds in [Set up a practice database](https://learning.nextia-ai.com/courses/sql/m01/set-up-a-practice-database/#build-the-database). The data is made up for this course; it has no real people.

In [ ]:
# Setup: download the Larkfield export, check it, and build larkfield.db in
# ticket-data/. Standard library only: the same steps as get_data.py.
import csv, hashlib, json, os, sqlite3, subprocess, sys, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/C04/"
FILES = {
    "customers.csv": "7b437642f0095524f8311f67a7caeb17100d23eed92618b9cf012cd3d3b79993",
    "tickets.csv": "9c47e08cf2ec5977733d32cc21eb97d0dd01b021d1f5fbf7ec31fef6279b1a28",
    "outcomes.csv": "67b4d9d9275b32061a8888050c351df7314054c11ee1ce29e856c1d5403d0653",
}
# The notebook works in its own folder ticket-data/, marked with a file, so it
# never writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
PROJECT = Path("ticket-data").resolve()
PROJECT.mkdir(exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def download(url, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

def fetch(name, expected):
    path = PROJECT / "data" / "raw" / name
    if not (path.exists() and hashlib.sha256(path.read_bytes()).hexdigest() == expected):
        download(LABS + "data/" + name, path)
        if hashlib.sha256(path.read_bytes()).hexdigest() != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder ticket-data and run again.")
    return path

DB = PROJECT / "larkfield.db"
if not DB.exists():
    paths = {name: fetch(name, sha) for name, sha in FILES.items()}
    build = sqlite3.connect(DB)
    build.executescript("""
        CREATE TABLE customers (customer_id TEXT, segment TEXT, region TEXT, joined_on TEXT);
        CREATE TABLE tickets (ticket_id TEXT, customer_id TEXT, created_at TEXT, channel TEXT, team TEXT,
            priority INTEGER, order_value REAL, word_count INTEGER, first_reply_minutes INTEGER,
            priority_now INTEGER, closed_at TEXT);
        CREATE TABLE outcomes (outcome_id TEXT, ticket_id TEXT, recorded_at TEXT, status TEXT,
            resolution_code TEXT, csat INTEGER);
        CREATE TABLE snapshot (name TEXT, value TEXT);
        INSERT INTO snapshot VALUES ('source', 'Larkfield help-desk export (synthetic, Nextia Learning C04)'),
            ('exported_at', '2026-07-01 06:00:00'), ('data_version', '1.0');
    """)
    for name, path in paths.items():
        with open(path, newline="", encoding="utf-8") as f:
            rows = list(csv.reader(f))
        marks = ", ".join("?" for _ in rows[0])
        build.executemany(f"INSERT INTO {name[:-4]} ({', '.join(rows[0])}) VALUES ({marks})",
                          [[None if v == "" else v for v in row] for row in rows[1:]])
    build.commit()
    build.close()

os.chdir(PROJECT)  # work in the project folder, as in the lesson
for folder in ("queries", "scripts", "reports"):
    Path(folder).mkdir(exist_ok=True)
# Read-only, as q.py: a query cannot change the data by mistake.
con = sqlite3.connect("file:larkfield.db?mode=ro", uri=True)
counts = [con.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0] for t in ("customers", "tickets", "outcomes")]
print("Ready: customers {}, tickets {}, outcomes {}".format(*counts))

## Setup: rebuild the earlier lessons' files

This lesson starts from files that you made in earlier lessons: the extract `data/extract/` from [Reproducible extraction](https://learning.nextia-ai.com/courses/sql/m03/reproducible-extraction/), the clean tables `data/clean/` from [Handle messy values](https://learning.nextia-ai.com/courses/sql/m04/handle-messy-values/), the modelling table `data/model/tickets_model.csv` from [Features and target timing](https://learning.nextia-ai.com/courses/sql/m05/features-and-target-timing/), the train, validation and test sets `data/model/` from [Choose a split](https://learning.nextia-ai.com/courses/sql/m05/choose-a-split/). The next cell downloads the course's finished scripts ([C04/pipeline](https://github.com/usmanahmed99/nextia-learning-labs/tree/main/C04/pipeline)) into `ticket-data` and runs them in order, so you have the same files here. It takes about 20 seconds.

You should see one line per step, ending with `split.py: done`. On your computer, your own scripts from those lessons make the same files.

In [ ]:
# Setup: rebuild the files of the earlier lessons with the course's scripts.
STEPS = ["extract", "clean", "build_features", "split"]
for file in ["queries/extract_tickets.sql", "queries/extract_outcomes.sql"] + [f"scripts/{s}.py" for s in STEPS]:
    if not Path(file).exists():
        download(LABS + "pipeline/" + file, Path(file))
for step in STEPS:
    result = subprocess.run([sys.executable, f"scripts/{step}.py"], capture_output=True, text=True)
    if result.returncode:
        raise SystemExit(f"{step}.py failed:\n{result.stdout}{result.stderr}")
    print(f"{step}.py: done")

## Setup: the helpers

The next cell defines small functions that the lesson uses. You do not need to read them, but you can.

- `sql(query)` runs a query and shows the result as a table, with the number of rows, like `q.py` and the boxes on the lesson page. It shows the first 20 rows; `sql(query, limit=None)` shows all of them.
- `check(query, ...)` compares the result of your query with the expected result, and says what is different. It does not show the answer.
- `expect(what, yours, expected)` compares one of your numbers with the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: show a query result, and check an answer without showing it.
try:
    import pandas as pd
except ImportError:
    pd = None

def run(query, db=None):
    cursor = (db or con).execute(query)
    columns = [d[0] for d in cursor.description] if cursor.description else []
    return columns, cursor.fetchall()

def sql(query, db=None, limit=20):
    """Run a query and show its result as a table, with the row count."""
    columns, rows = run(query, db)
    if not columns:
        print("No result: the cell has no SELECT yet. Write your query between the triple quotes, then run the cell again.")
        return
    shown = rows if limit is None else rows[:limit]
    if pd is not None:
        # Show NULL as NULL, as q.py and the boxes on the lesson page do.
        table = pd.DataFrame(shown, columns=columns, dtype=object)
        display(table.where(table.notna(), "NULL"))
    else:
        print("  ".join(columns))
        for row in shown:
            print("  ".join("NULL" if v is None else str(v) for v in row))
    more = f" (showing {len(shown)}; use limit=None to see all)" if len(shown) < len(rows) else ""
    print(f"{len(rows)} row{'s' if len(rows) != 1 else ''}{more}")

def fingerprint(rows, ordered=False):
    cells = [tuple("NULL" if v is None else f"{v:.6f}" if isinstance(v, float) else str(v) for v in r) for r in rows]
    return hashlib.sha256(repr(cells if ordered else sorted(cells)).encode()).hexdigest()[:16]

def check(query, rows, columns, expected, ordered=False, db=None):
    """Compare your result with the expected one: the column count, the row
    count, then the values (as a fingerprint, so the answer stays hidden)."""
    try:
        got_columns, got = run(query, db)
    except sqlite3.Error as error:
        print(f"Not yet: the query has an error: {error}")
        return
    if len(got_columns) != columns:
        print(f"Not yet: your result has {len(got_columns)} columns; the task asks for {columns}.")
    elif len(got) != rows:
        print(f"Not yet: your result has {len(got)} rows; the task expects {rows}.")
    elif fingerprint(got, ordered) != expected:
        print("Not yet: the number of rows is right, but some values" + (" or their order" if ordered else "") + " are different.")
    else:
        print("Check passed.")

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    same = abs(yours - expected) <= tolerance * max(1, abs(expected)) if isinstance(expected, float) else yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours}, which is not the expected value.")

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. Fit, then transform

A model needs numbers with no gaps. So the features go through **preprocessing** first: steps that fill missing values, put numbers on a similar scale and turn categories into numbers. Each step has two parts:

- **fit** learns numbers from data. A median imputer learns the median of each column. (**Imputation** means filling a missing value with a substitute.) A scaler learns the mean and the standard deviation (SD).
- **transform** uses the learned numbers to change data. The imputer fills each gap with the learned median. The scaler computes (value − mean) ÷ SD.

The next cell fits on five train values of `order_value`, one of them missing, and transforms two test values.

> **Predict.** The known train values are 20, 40, 60 and 100. What median will the imputer learn? Then run the cell.

In [ ]:
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

tiny_train = pd.DataFrame({"order_value": [20.0, 40.0, None, 60.0, 100.0]})
tiny_test = pd.DataFrame({"order_value": [None, 400.0]})

imputer = SimpleImputer(strategy="median").fit(tiny_train)
scaler = StandardScaler().fit(imputer.transform(tiny_train))
print("median learned from train:", imputer.statistics_)
print("mean and SD learned from train:", scaler.mean_.round(2), scaler.scale_.round(2))
print("test after transform:", scaler.transform(imputer.transform(tiny_test)).round(2).ravel())

> **Check.** You should see the median `[50.]`, the mean `[54.]`, the SD `[26.53]`, and the test values `[-0.15 13.04]`.

Check it by hand. The median is (40 + 60) ÷ 2 = 50. After the gap is filled, the mean is (20 + 40 + 50 + 60 + 100) ÷ 5 = 54. The missing test value becomes 50, and (50 − 54) ÷ 26.53 = −0.15. The value 400 becomes (400 − 54) ÷ 26.53 = 13.04: far above anything in train, and the transformed value says so.

Now the mistake: fit on the train **and** test values together. Run the cell.

In [ ]:
tiny_both = pd.concat([tiny_train, tiny_test])
imputer_all = SimpleImputer(strategy="median").fit(tiny_both)
scaler_all = StandardScaler().fit(imputer_all.transform(tiny_both))
print("median:", imputer_all.statistics_, "mean and SD:", scaler_all.mean_.round(2), scaler_all.scale_.round(2))
print("train after transform:", scaler_all.transform(imputer_all.transform(tiny_train)).round(2).ravel())

> **Check.** You should see the median `[60.]`, the mean `[105.71]` and the SD `[122.22]`. The train values become `[-0.7  -0.54 -0.37 -0.37 -0.05]`.

One test value moved every learned number. The preprocessing now contains information from the test set, before the test. That is leakage through preprocessing. The rule: **fit on the train set only, then transform the train, validation and test sets with the same learned numbers.** The [diagram on the lesson page](https://learning.nextia-ai.com/courses/sql/m05/fit-transformations-correctly/#fit-then-transform) shows it.

## 2. A pipeline keeps the rule for you

A **pipeline** joins the preprocessing steps and the model into one object. `pipeline.fit(train)` fits every step on the train rows. `pipeline.predict_proba(valid)` only transforms the validation rows and gives scores. A `ColumnTransformer` sends each list of columns to its own steps:

| Columns | Steps | What fit learns |
|---|---|---|
| Categories: `channel`, `team`, `segment`, `region` | `OneHotEncoder(handle_unknown="ignore")` | The list of categories in each column. |
| Numbers: `priority`, `order_value`, `word_count`, `customer_tenure_days`, `prior_tickets_90d`, `created_hour` | `SimpleImputer(strategy="median", add_indicator=True)`, then `StandardScaler()` | The median, then the mean and SD of each column. |

**One-hot encoding** turns one category column into several columns of 0 and 1, one for each category that fit saw. `add_indicator=True` adds a column of 0 and 1 for each number column that had gaps in train: 1 where the value was missing. A missing `order_value` has a meaning (an account ticket, no order), and the indicator keeps it.

Priya, the data scientist, asks for a quick check of the data with a simple model, a logistic regression. Run the next cell. It saves Mei's script as `scripts/model_check.py`.

In [ ]:
%%writefile scripts/model_check.py
"""Check the prepared data with a simple model: fit on train only, score on
valid. A check for the data, not a model for Larkfield (C05 trains models)."""

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

CATEGORIES = ["channel", "team", "segment", "region"]
NUMBERS = ["priority", "order_value", "word_count", "customer_tenure_days", "prior_tickets_90d", "created_hour"]
TARGET = "escalated_72h"


def build_pipeline():
    numbers = Pipeline([
        ("impute", SimpleImputer(strategy="median", add_indicator=True)),
        ("scale", StandardScaler()),
    ])
    prepare = ColumnTransformer([
        ("categories", OneHotEncoder(handle_unknown="ignore"), CATEGORIES),
        ("numbers", numbers, NUMBERS),
    ])
    return Pipeline([("prepare", prepare), ("model", LogisticRegression(max_iter=1000))])


def main():
    train = pd.read_csv("data/model/train.csv")
    valid = pd.read_csv("data/model/valid.csv")
    pipeline = build_pipeline()
    pipeline.fit(train[CATEGORIES + NUMBERS], train[TARGET])

    learned = pipeline.named_steps["prepare"].named_transformers_["numbers"]
    medians = learned["impute"].statistics_
    print("Medians learned from train:", {name: float(m) for name, m in zip(NUMBERS, medians)})
    channels = pipeline.named_steps["prepare"].named_transformers_["categories"].categories_[0]
    print("Channels learned from train:", [str(c) for c in channels])

    scores = pipeline.predict_proba(valid[CATEGORIES + NUMBERS])[:, 1]
    print("ROC AUC on valid:", round(roc_auc_score(valid[TARGET], scores), 3))


if __name__ == "__main__":
    main()

Run the script. `%run` runs a file, like `python scripts/model_check.py` in a terminal. It also keeps the script's names, such as `build_pipeline`, `CATEGORIES`, `NUMBERS` and `TARGET`, for the next cells.

**ROC AUC** in one sentence: the chance that the model scores a random escalated ticket higher than a random other ticket (0.5 is guessing, 1.0 is a perfect ranking). Course C05 explains it fully.

In [ ]:
%run scripts/model_check.py

> **Check.** You should see the median `86.21` for `order_value` and `3.0` for `prior_tickets_90d`, the channels `['chat', 'email', 'phone', 'unknown', 'web_form']`, and `ROC AUC on valid: 0.673`.

0.673 is a modest score: better than guessing, far from perfect. That is honest. The model learns from January to April and is tested on May, after the policy change.

## 3. A category that train never saw

Larkfield will soon answer messages on social media. The first ticket with `channel` = `social` will arrive, and no train ticket has that channel.

> **Predict.** What does the default encoder do with `social`? Then run the cell.

In [ ]:
from sklearn.preprocessing import OneHotEncoder

train = pd.read_csv("data/model/train.csv")
new_ticket = pd.DataFrame({"channel": ["social"]})

strict = OneHotEncoder().fit(train[["channel"]])
try:
    strict.transform(new_ticket)
except ValueError as error:
    print("ValueError:", error)

tolerant = OneHotEncoder(handle_unknown="ignore").fit(train[["channel"]])
print(tolerant.get_feature_names_out())
print(tolerant.transform(new_ticket).toarray())
print(tolerant.transform(pd.DataFrame({"channel": ["chat"]})).toarray())

> **Check.** You should see `ValueError: Found unknown categories ['social'] in column 0 during transform`. Then five channel columns, a row of five zeros for `social`, and a 1 in the first column for `chat`.

With the default, one new category stops the transform. In real use, the routing service would fail on the first social ticket. With `handle_unknown="ignore"`, the social ticket gets 0 in every channel column: "none of the channels I know". Do not "fix" this by fitting the encoder on all rows: that covers the categories in your file today, not the next new one.

## 4. Misconception: "fit the preprocessing on all the data"

A common mistake is to prepare the whole table first, and split it after. Then the medians, means and SDs come from every row, including validation and test. Mei fits the preprocessing on all 1,369 rows and the model on train only, as the mistake does.

> **Predict.** Will the score go up a lot, a little, or not at all? Then run the cell.

In [ ]:
import sys

import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

sys.path.insert(0, "scripts")
from model_check import CATEGORIES, NUMBERS, TARGET, build_pipeline

train = pd.read_csv("data/model/train.csv")
valid = pd.read_csv("data/model/valid.csv")
test = pd.read_csv("data/model/test.csv")
everything = pd.concat([train, valid, test])
features = CATEGORIES + NUMBERS

# The mistake: fit the preprocessing on every row, then the model on train.
prepare = build_pipeline().named_steps["prepare"]
prepare.fit(everything[features])
model = LogisticRegression(max_iter=1000).fit(prepare.transform(train[features]), train[TARGET])
scores = model.predict_proba(prepare.transform(valid[features]))[:, 1]
print("Median prior_tickets_90d:", float(prepare.named_transformers_["numbers"]["impute"].statistics_[4]))
print("ROC AUC on valid:", round(roc_auc_score(valid[TARGET], scores), 4))

# The correct way, for comparison, with four decimals.
pipeline = build_pipeline().fit(train[features], train[TARGET])
print("Train only, ROC AUC on valid:", round(roc_auc_score(valid[TARGET], pipeline.predict_proba(valid[features])[:, 1]), 4))

> **Check.** You should see `Median prior_tickets_90d: 5.0` (it is 3.0 from train only), `ROC AUC on valid: 0.6726` and `Train only, ROC AUC on valid: 0.6728`.

Be honest about this: **on this data, fitting a median imputer and a scaler on all rows changes the score very little.** A median or a mean of a feature carries almost no information about the target. Fix it anyway. The mistake grows with small datasets, rare categories and steps that learn from the target. The large inflations come from leaky features and leaky aggregates. The next section measures them.

## 5. Predict, then run: leaky features

The modelling table has no leaky columns. For the experiments, the next cell joins two of them back from the clean tickets: `priority_now` and `first_reply_minutes`. It also defines `auc_on_valid`: it fits a pipeline with the columns that you give it, on train only, and returns the ROC AUC on validation. Run the cell. It prints the honest score.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

leaks = pd.read_csv("data/clean/tickets.csv")[["ticket_id", "priority_now", "first_reply_minutes"]]
train = pd.read_csv("data/model/train.csv").merge(leaks, on="ticket_id", validate="one_to_one")
valid = pd.read_csv("data/model/valid.csv").merge(leaks, on="ticket_id", validate="one_to_one")
test = pd.read_csv("data/model/test.csv").merge(leaks, on="ticket_id", validate="one_to_one")


def pipeline_for(categories, numbers):
    """The same pipeline as build_pipeline(), for other lists of columns."""
    number_steps = Pipeline([("impute", SimpleImputer(strategy="median", add_indicator=True)),
                             ("scale", StandardScaler())])
    prepare = ColumnTransformer([("categories", OneHotEncoder(handle_unknown="ignore"), categories),
                                 ("numbers", number_steps, numbers)])
    return Pipeline([("prepare", prepare), ("model", LogisticRegression(max_iter=1000))])


def auc_on_valid(categories, numbers, train, valid):
    """Fit on train only, then return the ROC AUC on valid, rounded to 3 decimals."""
    pipeline = pipeline_for(categories, numbers).fit(train[categories + numbers], train[TARGET])
    scores = pipeline.predict_proba(valid[categories + numbers])[:, 1]
    return round(float(roc_auc_score(valid[TARGET], scores)), 3)


print(len(train), len(valid), len(test), "rows")
print("ROC AUC on valid, honest features:", auc_on_valid(CATEGORIES, NUMBERS, train, valid))

> **Check.** You should see `927 227 215 rows` and `ROC AUC on valid, honest features: 0.673`.

Tomás, the AI engineer, wants to add `priority_now`, the current priority. The help desk sets it to 1 when a ticket is escalated.

> **Predict.** What ROC AUC do you expect with `priority_now` as an extra number? Write your prediction in the cell, between the quotes. Then run it.

In [ ]:
my_prediction = ""  # for example "about 0.7" or "above 0.9"

print("your prediction:", my_prediction or "(none)")
print("ROC AUC on valid with priority_now:", auc_on_valid(CATEGORIES, NUMBERS + ["priority_now"], train, valid))

> **Check.** You should see `0.964`, from 0.673.

The column does not predict the escalation; it records it. When a new ticket arrives, `priority_now` is the same as `priority`, and the 0.964 disappears. This is **leakage**: information in the features that the model would not have at prediction time.

Run the next cell. It adds `first_reply_minutes` instead, the minutes until the first reply. It is not the target, but it is also written after creation.

In [ ]:
print("ROC AUC on valid with first_reply_minutes:", auc_on_valid(CATEGORIES, NUMBERS + ["first_reply_minutes"], train, valid))

> **Check.** You should see `0.856`. Escalated tickets get a fast first reply from a senior agent, so the reply time is a trace of the escalation.

The last leak is a **leaky aggregate**: each customer's escalation rate, computed over **all** rows, including validation and test. It looks like a customer fact. Run the cell.

In [ ]:
everything = pd.concat([train, valid, test])
customer_rate_all = everything.groupby("customer_id")[TARGET].mean()
rate_train = train.assign(customer_rate_all=train["customer_id"].map(customer_rate_all))
rate_valid = valid.assign(customer_rate_all=valid["customer_id"].map(customer_rate_all))
print("ROC AUC on valid with customer_rate_all:", auc_on_valid(CATEGORIES, NUMBERS + ["customer_rate_all"], rate_train, rate_valid))

> **Check.** You should see `0.767`. A May ticket's rate includes that ticket's own target, and the targets of later tickets. The model reads part of the answer from it.

| Features (fit on train, score on May) | ROC AUC on valid |
|---|---|
| The 10 honest features | **0.673** |
| + `priority_now` | **0.964** |
| + `first_reply_minutes` | **0.856** |
| + each customer's escalation rate over all rows | **0.767** |
| Honest features, preprocessing fit on all rows | 0.6726 |

## 6. Your turn: mean instead of median

Change one thing: the imputer's `strategy`. `set_params` changes one setting of a pipeline step without rewriting the script. The name `prepare__numbers__impute__strategy` is the path to it: the step `prepare`, its branch `numbers`, its step `impute`, the setting `strategy`.

> **Predict.** With the mean, will the value that fills a missing `order_value` go up or down? Will the score change much?

> **Your turn.** Set `strategy` to `"mean"`, run the cell, then run the check cell.

In [ ]:
strategy = "mean"  # change this

variant = build_pipeline().set_params(prepare__numbers__impute__strategy=strategy)
variant.fit(train[CATEGORIES + NUMBERS], train[TARGET])
fill = variant.named_steps["prepare"].named_transformers_["numbers"]["impute"].statistics_
fill_order_value = round(float(fill[NUMBERS.index("order_value")]), 2)
variant_auc = round(float(roc_auc_score(valid[TARGET], variant.predict_proba(valid[CATEGORIES + NUMBERS])[:, 1])), 3)
print("order_value fill value:", fill_order_value)
print("ROC AUC on valid:", variant_auc)

In [ ]:
if fill_order_value == 86.21:
    print("Not yet: the fill value is still the median. Set strategy to \"mean\" and run the cell again.")
elif fingerprint([[fill_order_value, variant_auc]]) == "049b8c1b09649d92":
    print("Check passed.")
else:
    print("Not yet: change only the strategy, to \"mean\", and keep the rest of the cell.")

The fill value for `order_value` goes **up**, from 86.21 to 120.30. A few large orders pull the mean up, but not the median. The score hardly changes: 0.672 instead of 0.673. The choice still matters for the meaning: a filled account ticket now "has" an order of $120.30. In Module 4, you decided that a missing order value is not an unknown number. So the median plus an indicator is the more honest choice here.

## 7. Failure case: `fit_transform` on the validation set

Tomás writes the preprocessing and the model as separate steps, without a pipeline. Then he prepares the validation set in the same way as the train set.

> **Predict.** What happens when he scores the validation set? Then run the cell.

In [ ]:
prepare = build_pipeline().named_steps["prepare"]
train_x = prepare.fit_transform(train[features])
model = LogisticRegression(max_iter=1000).fit(train_x, train[TARGET])
valid_x = prepare.fit_transform(valid[features])  # fit again, on valid
try:
    model.predict_proba(valid_x)
except ValueError as error:
    print("ValueError:", error)

> **Check.** You should see `ValueError: X has 27 features, but LogisticRegression is expecting 28 features as input.`

**Cause:** `fit_transform` on the validation set fits the preprocessing again, on the May tickets. The May tickets have no customer with the segment `unknown`, so the encoder learned one category fewer. The error is the lucky case. If May had the same categories, the code would run, but the scaler would use the May mean and SD. The same value would mean something different in train and validation, and nobody would see an error.

**Fix:** `fit` or `fit_transform` only on the train set; only `transform` on everything else. Better, use one pipeline, as `model_check.py` does: then there is only one `fit` call, and it gets the train rows.

## 8. Check yourself

### Exercise 1: the final test score

Priya is done choosing. Now she fits the honest pipeline once on train **and** validation together, and scores the test set (June) once.

> **Your turn.** In the next cell, fit `final_pipeline` on `train_and_valid` with `CATEGORIES + NUMBERS`. Then put its ROC AUC on `test`, rounded to 3 decimals, in `test_auc`. Run the cell, then the check cell.

In [ ]:
train_and_valid = pd.concat([train, valid])
final_pipeline = build_pipeline()
final_pipeline.fit(train_and_valid[CATEGORIES + NUMBERS], train_and_valid[TARGET])
test_scores = final_pipeline.predict_proba(test[CATEGORIES + NUMBERS])[:, 1]
test_auc = round(roc_auc_score(test[TARGET], test_scores), 3)
print(test_auc)

In [ ]:
if test_auc is ...:
    print("Not yet: replace ... with your code.")
elif not hasattr(final_pipeline.named_steps["model"], "coef_"):
    print("Not yet: fit final_pipeline before you score the test set.")
elif fingerprint([[round(float(test_auc), 3)]]) == "81f92cb441672ab0":
    print("Check passed.")
else:
    print("Not yet: fit on train_and_valid (1,154 rows), score on test (215 rows), and round to 3 decimals.")

The test score, 0.761, is higher than the validation score, 0.673. The final model also learned from May, after the policy change, so it knows the new pattern. Nobody looked at the test set before this cell.

### Exercise 2: find the seeded leak

A colleague offers three new columns. One of them leaks. Run the next cell first: it adds the three columns to the train and validation rows. Each one is a category.

- `weekday`: the day of the week when the ticket was created.
- `order_size`: `small`, `medium`, `large` or `none`, from `order_value`.
- `reply_speed`: `fast` or `slow`.

In [ ]:
def add_offered_columns(df):
    created = pd.to_datetime(df["created_at"])
    return df.assign(
        weekday=created.dt.day_name(),
        order_size=pd.cut(df["order_value"], [0, 50, 150, float("inf")], labels=["small", "medium", "large"])
        .cat.add_categories("none").fillna("none").astype("str"),
        reply_speed=(df["first_reply_minutes"] < 30).map({True: "fast", False: "slow"}),
    )


seeded_train = add_offered_columns(train)
seeded_valid = add_offered_columns(valid)
seeded_train[["weekday", "order_size", "reply_speed"]].head(3)

> **Check.** You should see three rows with a day name, an order size and a reply speed.

> **Your turn.** For each of the three columns, add it alone to the honest features, and compute the ROC AUC on validation with `auc_on_valid`. Put the three scores in the dictionary `scores_by_column`. Then set `leaky_column` to the name of the column that leaks. Use the scores as a clue, then confirm with a question: **when is the value written?**

In [ ]:
scores_by_column = {}
for column in ["weekday", "order_size", "reply_speed"]:
    scores_by_column[column] = auc_on_valid(CATEGORIES + [column], NUMBERS, seeded_train, seeded_valid)
print(scores_by_column)

leaky_column = "reply_speed"

In [ ]:
if not all(isinstance(v, float) for v in scores_by_column.values()):
    print("Not yet: each value in scores_by_column must be a ROC AUC from auc_on_valid.")
elif leaky_column not in scores_by_column:
    print('Not yet: set leaky_column to "weekday", "order_size" or "reply_speed".')
elif leaky_column != max(scores_by_column, key=scores_by_column.get):
    print("Not yet: look again at the three scores. Which column lifts the score far above 0.673?")
elif fingerprint([[leaky_column]]) == "8950ac5841475aae":
    print("Check passed.")
else:
    print("Not yet: ask when each value is written.")

`reply_speed` comes from `first_reply_minutes`, which is written after the first reply, so it is not known when the ticket arrives. A new name and a coarser form do not remove a leak. `weekday` and `order_size` are known at creation, and they change the score very little.

In the lesson's module check, you explain how this leak inflated the evaluation. Write two or three sentences in the next cell now, while the numbers are in front of you.

*Your explanation here (double-click to edit):*

## 9. Module practice: plan a new model

Grace, the support lead, has a new question. When an agent resolves a ticket, can a model say if the customer will reopen it within 14 days? Plan the data for that model. **Do not fit a model.** The [lesson page](https://learning.nextia-ai.com/courses/sql/m05/fit-transformations-correctly/#module-practice) has Mei's plan for the escalation model as the worked example.

> **Your turn.** Fill in the table in the next cell (double-click to edit). For every decision, write one sentence of reason. Look carefully at the prediction time: it is no longer the creation of the ticket. Then compare with the model answer on the lesson page.

| Question | Decision | Reason |
|---|---|---|
| Prediction time | | |
| Target | | |
| Target maturity | | |
| Features | | |
| Never features | | |
| Split | | |
| Preprocessing | | |
| Checks before any model | | |

## Check your understanding and recap

Answer the **knowledge checks** and the **module check** on the [lesson page](https://learning.nextia-ai.com/courses/sql/m05/fit-transformations-correctly/#module-check). They count toward your certificate when you are signed in and enrolled.

In this lesson you learned that fit learns numbers and transform uses them. Fit on the train set only, inside one pipeline. `handle_unknown="ignore"` keeps a new category from stopping the model. On this data, fitting the preprocessing on all rows changed the score only in the fourth decimal. The large inflations came from leaks: 0.673 became 0.964 with `priority_now`, 0.856 with `first_reply_minutes` and 0.767 with a customer rate over all rows.

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Validation rules](https://learning.nextia-ai.com/courses/sql/m06/validation-rules/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/sql/m05/fit-transformations-correctly/)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.